# 用 Sciverse 构建科研文献综述 Agent

> 从一句研究问题出发，自动检索、摘要、生成带引用的文献综述

---

**Sciverse Cookbook** | [文档首页](https://sciverse.space/docs#cookbook) | [申请 API Key](https://sciverse.space/docs#auth)

## 前置准备

1. 在 [Sciverse 控制台](https://sciverse.space/) 申请 API Token
2. 安装依赖：`pip install httpx anthropic`
3. 将下方 `sv-...` 替换为你的真实 Token


In [ ]:
# 安装依赖（Colab 环境）
!pip install -q httpx anthropic


## Step 1: 语义检索相关片段

使用 semantic_search 获取与研究问题相关的文献片段


In [ ]:
import httpx

BASE = "https://api.sciverse.space"
TOKEN = "sv-..."

async def search_literature(query: str, top_k: int = 20):
    async with httpx.AsyncClient() as client:
        resp = await client.post(
            f"{BASE}/agentic-search",
            headers={"Authorization": f"Bearer {TOKEN}"},
            json={"query": query, "top_k": top_k}
        )
        return resp.json()["hits"]

hits = await search_literature(
    "Transformer applications in protein structure prediction 2020-2024"
)
print(f"Found {len(hits)} relevant chunks")

## Step 2: 读取原文上下文

对高分片段调用 read_content 获取更完整的上下文


In [ ]:
async def read_context(doc_id: str, offset: int = 0, limit: int = 4096):
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"{BASE}/content",
            headers={"Authorization": f"Bearer {TOKEN}"},
            params={"doc_id": doc_id, "offset": offset, "limit": limit}
        )
        return resp.json()

# 对 top 5 高分片段读取上下文
evidences = []
for hit in sorted(hits, key=lambda x: x["score"], reverse=True)[:5]:
    ctx = await read_context(hit["doc_id"], hit.get("offset", 0))
    evidences.append({
        "title": hit["title"],
        "doc_id": hit["doc_id"],
        "chunk": hit["chunk"],
        "context": ctx["content"],
        "score": hit["score"]
    })

## Step 3: 生成带引用的综述

将证据传给 LLM 生成结构化综述


In [ ]:
from anthropic import Anthropic

client = Anthropic()

evidence_text = "\n\n".join([
    f"[{e['doc_id']}] {e['title']}\n{e['context']}"
    for e in evidences
])

msg = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=4096,
    messages=[{
        "role": "user",
        "content": f"""基于以下文献证据，生成一份关于 Transformer 在蛋白质结构预测中应用的综述。
每个论点必须标注来源 [doc_id]。

{evidence_text}"""
    }]
)
print(msg.content[0].text)

---

## 下一步

- 📖 [查看完整 API 文档](https://sciverse.space/docs#sciverse/api)
- 🔑 [申请 / 管理 API Token](https://sciverse.space/)
- 📚 [浏览更多 Cookbook 案例](https://sciverse.space/docs#cookbook)
- 💬 [加入开发者社群](https://sciverse.space/)
